# Data Cleaning and Preprocessing on Titanic Dataset

**Objective:**
Transform a raw dataset into a clean, analysis-ready dataset by identifying and fixing common data quality issues such as missing values, duplicates, inconsistent formats, incorrect data types, and outliers.

**Tools Used**
- Python
- Pandas
- NumPy
- Jupyter Notebook

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Initial Dataset Information
The first step is to inspect the dataset structure, identify missing values, duplicates, incorrect data types, and potential anomalies.

In [3]:
print("Shape:", df.shape)

print("\nData Types")
print(df.dtypes)

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
display(df.describe(include="all"))

Shape: (891, 12)

Data Types
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Missing Values
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Duplicate Rows
0

Summary Statistics


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,B96 B98,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [4]:
print("Age Range:", df["Age"].min(), "-", df["Age"].max())
print("Fare Range:", df["Fare"].min(), "-", df["Fare"].max())

Age Range: 0.42 - 80.0
Fare Range: 0.0 - 512.3292


## Data Quality Report

### Missing Values
- Age contains missing values.
- Cabin contains many missing values.
- Embarked contains a few missing values.

### Duplicate Rows
No duplicate rows are present.

### Data Types
Most columns already have appropriate data types.

### Value Ranges
Age and Fare contain unusually large values that may be potential outliers.

In [6]:
df_clean = df.copy()

df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())

df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
# Cabin -> Drop column
df_clean.drop(columns=["Cabin"], inplace=True)

## Missing Value Strategy

### Age
Filled using the **median** because Age is numerical and contains outliers.

### Embarked
Filled using the **mode** because it is a categorical feature.

### Cabin
Dropped because more than 75% of its values are missing, making it unreliable for analysis.

In [7]:
before = len(df_clean)

df_clean.drop_duplicates(inplace=True)

after = len(df_clean)

print("Duplicates Removed:", before-after)

Duplicates Removed: 0


## Duplicate Removal

Duplicate rows were checked and removed.

In this dataset, no duplicate rows were found.

In [8]:
df_clean["Sex"] = df_clean["Sex"].str.strip().str.capitalize()

df_clean["Embarked"] = df_clean["Embarked"].str.upper()

## Standardization

The categorical columns were standardized to maintain consistent formatting.

Examples:
- male → Male
- female → Female

In [9]:
Q1 = df_clean["Fare"].quantile(0.25)
Q3 = df_clean["Fare"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

outliers = df_clean[(df_clean["Fare"]<lower) | (df_clean["Fare"]>upper)]

print("Fare Outliers:", len(outliers))

Fare Outliers: 116


In [10]:
df_clean["Fare"] = np.where(df_clean["Fare"]>upper, upper, df_clean["Fare"])
df_clean["Fare"] = np.where(df_clean["Fare"]<lower, lower, df_clean["Fare"])

## Outlier Handling

The IQR method was used to detect outliers.

The Fare column contained several extreme values.

Instead of deleting those rows, the values were capped using the upper and lower IQR limits to preserve the dataset while reducing the impact of extreme observations.

In [11]:
df_clean["PassengerId"] = df_clean["PassengerId"].astype(str)

df_clean["Fare"] = df_clean["Fare"].astype(float)

df_clean["Age"] = df_clean["Age"].astype(float)

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    object 
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 76.7+ KB


## Data Type Correction

- PassengerId → String
- Age → Float
- Fare → Float

The data types are now appropriate for analysis.

In [12]:
summary = pd.DataFrame({
    "Metric":[
        "Rows",
        "Columns",
        "Missing Values",
        "Duplicate Rows"
    ],
    "Before":[
        len(df),
        len(df.columns),
        df.isnull().sum().sum(),
        df.duplicated().sum()
    ],
    "After":[
        len(df_clean),
        len(df_clean.columns),
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum()
    ]
})

summary

,Metric,Before,After
0,Rows,891,891
1,Columns,12,11
2,Missing Values,866,0
3,Duplicate Rows,0,0


## Before vs After Summary

The cleaning process successfully:

- Reduced missing values
- Removed unnecessary columns
- Standardized categorical values
- Corrected data types
- Handled outliers
- Produced a clean, analysis-ready dataset

In [14]:
df_clean.to_csv("Titanic_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


# Conclusion

The Titanic dataset was successfully cleaned by handling missing values, removing duplicate records, standardizing categorical data, correcting data types, and treating outliers.

The resulting dataset is consistent, reliable, and ready for exploratory data analysis, visualization, and machine learning tasks.